In [ ]:
from googleapiclient.discovery import build
from datetime import datetime
import pandas as pd
import time

# Replace with your actual API key
api_key = ''
youtube = build('youtube', 'v3', developerKey=api_key)

def search_top_videos(query, max_results=10):
    videos = []
    next_page_token = None
    while len(videos) < max_results:
        response = youtube.search().list(
            q=query,
            part='id,snippet',
            type='video',
            maxResults=50,
            order='relevance',
            pageToken=next_page_token
        ).execute()

        for item in response['items']:
            if item['id']['kind'] == 'youtube#video':
                videos.append({
                    'video_id': item['id']['videoId'],
                    'video_title': item['snippet']['title']
                })
                if len(videos) >= max_results:
                    break

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break
    return videos

def get_video_stats(video_id):
    response = youtube.videos().list(
        part='statistics',
        id=video_id
    ).execute()
    stats = response['items'][0]['statistics']
    like_count = int(stats.get('likeCount', 0))
    view_count = int(stats.get('viewCount', 0))
    return like_count, view_count

def get_replies(comment_id, max_results=50):
    replies = []
    next_page_token = None
    while True:
        response = youtube.comments().list(
            part='snippet',
            parentId=comment_id,
            maxResults=min(100, max_results - len(replies)),
            pageToken=next_page_token,
            textFormat='plainText'
        ).execute()

        for item in response['items']:
            snippet = item['snippet']
            replies.append({
                'reply_id': item['id'],
                'reply_text': snippet['textDisplay'],
                'reply_author': snippet.get('authorDisplayName', 'N/A'),
                'reply_published': snippet['publishedAt']
            })

        next_page_token = response.get('nextPageToken')
        if not next_page_token or len(replies) >= max_results:
            break
    return replies

def get_comments_sorted_with_replies(video_id, video_title, like_count, view_count, max_total=500, top_n=100):
    comments_data = []
    next_page_token = None

    while len(comments_data) < max_total:
        response = youtube.commentThreads().list(
            part='snippet',
            videoId=video_id,
            maxResults=min(100, max_total - len(comments_data)),
            pageToken=next_page_token,
            textFormat='plainText'
        ).execute()

        for item in response['items']:
            snippet = item['snippet']['topLevelComment']['snippet']
            comment_id = item['snippet']['topLevelComment']['id']
            comment_text = snippet['textDisplay']
            author = snippet.get('authorDisplayName', 'N/A')
            timestamp = snippet['publishedAt']
            timestamp = datetime.strptime(timestamp, "%Y-%m-%dT%H:%M:%SZ")
            like_count_comment = snippet.get('likeCount', 0)
            total_reply_count = item['snippet']['totalReplyCount']

            comments_data.append({
                'video_id': video_id,
                'video_title': video_title,
                'video_like_count': like_count,
                'video_view_count': view_count,
                'comment_id': comment_id,
                'author': author,
                'comment': comment_text,
                'timestamp': timestamp,
                'comment_like_count': like_count_comment,
                'reply_count': total_reply_count
            })

        next_page_token = response.get('nextPageToken')
        if not next_page_token:
            break

    # Sort by likes, then replies
    comments_data.sort(key=lambda x: (x['comment_like_count'], x['reply_count']), reverse=True)
    top_comments = comments_data[:top_n]

    # Fetch replies for each top comment
    enriched = []
    for comment in top_comments:
        replies = get_replies(comment['comment_id'], max_results=50) if comment['reply_count'] > 0 else []
        comment['replies'] = replies
        enriched.append(comment)

    return enriched

# Main logic
all_rows = []
for year in range(2025, 2014, -1):
    query = f'best countries to visit in {year}'
    print(f"Searching for: {query}")
    videos = search_top_videos(query, max_results=10)

    for video in videos:
        print(f"Processing video: {video['video_title']}")
        try:
            like_count, view_count = get_video_stats(video['video_id'])
            comments = get_comments_sorted_with_replies(
                video_id=video['video_id'],
                video_title=video['video_title'],
                like_count=like_count,
                view_count=view_count,
                max_total=500,
                top_n=100
            )
            for comment in comments:
                if comment['replies']:
                    for reply in comment['replies']:
                        all_rows.append({
                            **{k: v for k, v in comment.items() if k != 'replies'},
                            'reply_id': reply['reply_id'],
                            'reply_text': reply['reply_text'],
                            'reply_author': reply['reply_author'],
                            'reply_published': reply['reply_published']
                        })
                else:
                    all_rows.append({
                        **{k: v for k, v in comment.items() if k != 'replies'},
                        'reply_id': None,
                        'reply_text': None,
                        'reply_author': None,
                        'reply_published': None
                    })

        except Exception as e:
            print(f"Error processing video {video['video_id']}: {e}")
        time.sleep(1)

df = pd.DataFrame(all_rows)
df.to_csv('top_comments_2015_to_2025_with_replies.csv', index=False)
print(f"Done: {len(df)} rows saved to CSV.")


Searching for: best countries to visit in 2025
Processing video: Top 15 Countries to Visit in 2025 | Ultimate Travel Guide
Processing video: Top 10 Places To Visit in 2025 (Year of Travel)
Processing video: Top 10 Best Countries To Visit In 2025 #top10 #countries #visit #2025
Processing video: Best Countries to Visit in 2025 (by Month)
Processing video: 30 Best Countries To Visit In 2025 | Travel Video 4K
Processing video: 25 Best Places To Visit In 2025 | Travel Guide 2025
Processing video: Best places to visit in 2025 by month 🌍 2025 travel destinations
Processing video: 10 Best Countries to Visit for Long Term Stays in 2025
Processing video: Best place to visit in Singapore 2025 #2025shorts #2025 #shorts #shortfeed
Processing video: 50 Best Places to Visit in 2025 | Travel Guide
Searching for: best countries to visit in 2024
Processing video: 25 Best Countries To Visit In 2024 | Travel Guide 2024
Processing video: The Top 5 Countries YOU Should Visit in 2024! ✈️ 🌍 #travel #travelvlo